In [ ]:
import os
os.chdir("..")      # on se place à la racine du projet
print(os.getcwd())

In [ ]:
from src.dataset import build_cache
CLASSES = ["chair", "table", "toilet"]
for s in ["train", "test"]:
    build_cache("data/raw/ModelNet10", CLASSES, s, n=1024, max_per_class=300)

Entraînement

In [ ]:
import yaml, torch, pandas as pd
from pathlib import Path
from torch.utils.data import DataLoader
from src.dataset import PointCloudDataset
from src.models import PointNetAE
from src.losses import chamfer

cfg = yaml.safe_load(open("configs/baseline.yaml"))
torch.manual_seed(cfg["seed"])
dev = "cuda" if torch.cuda.is_available() else "cpu"

tr = DataLoader(PointCloudDataset("data/processed/train_1024.npz"), cfg["batch_size"], shuffle=True, drop_last=True)
te = DataLoader(PointCloudDataset("data/processed/test_1024.npz"), cfg["batch_size"])

model = PointNetAE(cfg["latent"], cfg["n_points"]).to(dev)
opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
hist = []

for ep in range(cfg["epochs"]):
    model.train(); trl = 0
    for x, _ in tr:
        x = x.to(dev)
        rec, _ = model(x)
        loss = chamfer(rec, x).mean()
        opt.zero_grad(); loss.backward(); opt.step()
        trl += loss.item()
    model.eval(); tel = 0
    with torch.no_grad():
        for x, _ in te:
            x = x.to(dev)
            tel += chamfer(model(x)[0], x).mean().item()
    hist.append((ep, trl / len(tr), tel / len(te)))
    if ep % 10 == 0:
        print(ep, hist[-1])

Path("checkpoints").mkdir(exist_ok=True)
torch.save(model.state_dict(), "checkpoints/baseline_ae.pt")
pd.DataFrame(hist, columns=["epoch", "train_cd", "test_cd"]).to_csv("results/tables/02_training_log.csv", index=False)

courbe

In [ ]:
import matplotlib.pyplot as plt
h = pd.read_csv("results/tables/02_training_log.csv")
plt.plot(h.epoch, h.train_cd, label="train")
plt.plot(h.epoch, h.test_cd, label="test")
plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("Chamfer"); plt.legend()
plt.savefig("results/figures/02_loss_curve.png", dpi=150, bbox_inches="tight")
plt.show()

Evaluer (chiffre + images)

In [ ]:
import matplotlib.pyplot as plt

torch.manual_seed(0)
model.eval()

# 2 objets aléatoires par classe (3 classes -> 6 colonnes)
ds = PointCloudDataset("data/processed/test_1024.npz")
idx = []
for c in range(3):
    ids = (ds.y == c).nonzero().flatten()
    pick = ids[torch.randperm(len(ids))[:2]]
    idx += pick.tolist()

x = torch.stack([ds[i][0] for i in idx]).to(dev)
with torch.no_grad():
    rec, _ = model(x)

fig = plt.figure(figsize=(12, 4))
for i in range(6):
    for r, pts in enumerate([x[i], rec[i]]):
        ax = fig.add_subplot(2, 6, r * 6 + i + 1, projection="3d")
        p = pts.cpu().numpy()
        ax.scatter(p[:, 0], p[:, 1], p[:, 2], s=1)
        ax.set_axis_off()
fig.suptitle("Haut : originaux / Bas : reconstructions")
fig.savefig("results/figures/02_reconstructions.png", dpi=150, bbox_inches="tight")
plt.show()

# Chamfer sur tout le jeu de test
with torch.no_grad():
    cd = torch.cat([chamfer(model(a.to(dev))[0], a.to(dev)) for a, _ in te]).mean().item()
print("Chamfer test :", cd)
pd.DataFrame([{"model": "PointNet-AE", "latent": 128, "test_chamfer": cd}]).to_csv(
    "results/tables/02_baseline_metrics.csv", index=False)

Générer de nouvelles formes 

In [ ]:
with torch.no_grad():
    Z = torch.cat([model.encode(a.to(dev)) for a, _ in tr]).cpu()
mu = Z.mean(0)
cov = torch.cov(Z.T) + 1e-4 * torch.eye(Z.shape[1])
dist = torch.distributions.MultivariateNormal(mu, cov)

z = dist.sample((12,)).to(dev)
with torch.no_grad():
    gen = model.decode(z).cpu().numpy()

fig = plt.figure(figsize=(12, 4))
for i in range(12):
    ax = fig.add_subplot(2, 6, i + 1, projection="3d")
    ax.scatter(gen[i, :, 0], gen[i, :, 1], gen[i, :, 2], s=1)
    ax.set_axis_off()
fig.savefig("results/figures/02_generated_samples.png", dpi=150, bbox_inches="tight")